<a href="https://colab.research.google.com/github/jsemprini/Iowa-Water-Nitrate-Births-8488update/blob/main/Copy_of_1_createwater_final_T1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# File 1 (Final V3) — Create the Iowa county-quarter nitrate exposure panel

**Purpose.** Build and validate the completed Iowa county-quarter nitrate panel used to construct the final first-trimester exposure.

### Final design decisions

- The water reconstruction preserves the validated  approach, including the complete 1982Q1–1988Q4 county-quarter grid, spatial predictor, log-scale imputation model, Duan smearing retransformation, and all cross-validation and quality-control procedures.
- The analytic birth cohort begins at gestational start on/after **1984-01-01**; 1982 remains in the water reconstruction to preserve the V3 estimation/validation panel but is not itself an eligible T1 start period.
- Nitrate values at or above 10 mg/L are **never deleted or truncated**. Births with T1 nitrate >=10 will be excluded after linkage.
- Water-sample coordinates are assigned to Iowa counties using point-in-polygon, with audited nearest-county fallback if needed.
- Public-water-system measurements are collapsed hierarchically to county-quarter exposure. Population-served weighting is used only if population coverage meets the prespecified threshold; otherwise the county median of PWS-quarter medians is used.
- Missing county-quarters are predicted with the model: `log1p(nitrate) ~ county FE + year-quarter FE + log1p(spatial nitrate)`, with Duan smearing on retransformation.
- Validation retains random 5-fold cross-validation, blocked contiguous-quarter validation, and artificial missingness-pattern validation. Spatial predictors are rebuilt from training data only during validation to prevent leakage.
- Both observed nitrate and the completed series are retained in the water panel because File 3 constructs both `t1_mean_observed` and `t1_mean_complete`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


Mounted at /content/drive


In [ ]:
# ============================================================
# 0. SETTINGS
# ============================================================
import os
import gc
import numpy as np
import pandas as pd
import geopandas as gpd
import statsmodels.formula.api as smf
from scipy.spatial.distance import cdist
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option('display.max_columns', 250)
pd.set_option('display.width', 260)

PANEL_START_YEAR = 1982       # retained to preserve the validated V3 water reconstruction
PRIMARY_COHORT_START_YEAR = 1983
PANEL_END_YEAR = 1988
N_QUARTERS_PER_COUNTY = (PANEL_END_YEAR - PANEL_START_YEAR + 1) * 4

RAW_WATER_PATH = (
    '/content/drive/MyDrive/Current Research/Water/water-raw/'
    'original-finished_clean.csv'
)
COUNTY_SHP_PATH = (
    '/content/drive/MyDrive/Current Research/Water/water-raw/'
    'cb_2018_us_county_500k.shp'
)
OUTPUT_DIR = '/content/drive/MyDrive/plos-update-v3/1-water'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Use population weights only if coverage is essentially complete.
POPULATION_WEIGHT_COVERAGE_MIN = 0.95

# Validation settings. Increase repetitions for the final frozen run if desired.
BLOCKED_CV_REPS = 50
MISSINGNESS_REPS = 30
MISSINGNESS_RETENTION_RATES = [0.75, 0.50, 0.25]
WELL_OBSERVED_MIN_QUARTERS_1983_1988 = 18
RANDOM_SEED = 12345

print('Water panel:', f'{PANEL_START_YEAR}Q1-{PANEL_END_YEAR}Q4')
print('Primary pregnancy cohort begins in:', PRIMARY_COHORT_START_YEAR)
print('Output:', OUTPUT_DIR)


Water panel: 1982Q1-1988Q4
Primary pregnancy cohort begins in: 1983
Output: /content/drive/MyDrive/plos-update-v3/1-water


In [ ]:
# ============================================================
# 1. LOAD AND AUDIT RAW WATER DATA
# ============================================================
df = pd.read_csv(RAW_WATER_PATH, low_memory=False)
print(f'Raw observations: {len(df):,}')
print('Raw columns:', list(df.columns))

required = ['date_taken', 'nitrate', 'Latitude', 'Longitude', 'pws_id']
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise KeyError('Required raw-water columns missing: ' + ', '.join(missing_required))

df['date_taken'] = pd.to_datetime(df['date_taken'], errors='coerce')
df['nitrate'] = pd.to_numeric(df['nitrate'], errors='coerce')
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

if df['date_taken'].isna().any():
    raise ValueError('Invalid sampling dates detected.')
if df['nitrate'].isna().any():
    raise ValueError('Missing nitrate measurements detected.')
if (df['nitrate'] < 0).any():
    raise ValueError('Negative raw nitrate values detected; inspect source coding before proceeding.')
if df[['Latitude', 'Longitude']].isna().any().any():
    raise ValueError('Missing coordinates detected.')

bad_coords = ~(
    df['Latitude'].between(40.3, 43.6)
    & df['Longitude'].between(-96.8, -90.0)
)
print(f'Coordinates outside expected Iowa bounding box: {bad_coords.sum():,}')
print('\nRaw nitrate distribution:')
print(df['nitrate'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))
print('Raw nitrate >=10 mg/L:', int((df['nitrate'] >= 10).sum()))

# Audit likely units/sample-type fields if they exist. We do not silently recode them.
for candidate in ['units', 'unit', 'result_unit', 'sample_type', 'sampletype', 'sample_point_type', 'source_type']:
    if candidate in df.columns:
        vals = df[candidate].astype('string').value_counts(dropna=False).head(30)
        print(f'\n{candidate} values:')
        print(vals)

# Detect a population-served variable, but use it only if coverage is high enough later.
POPULATION_CANDIDATES = [
    'population_served', 'PopulationServed', 'pop_served', 'population',
    'pws_population', 'popserv', 'populationserved'
]
POPULATION_COLUMN = next((c for c in POPULATION_CANDIDATES if c in df.columns), None)
if POPULATION_COLUMN is not None:
    df['population_served'] = pd.to_numeric(df[POPULATION_COLUMN], errors='coerce')
    df.loc[df['population_served'] <= 0, 'population_served'] = np.nan
    print('\nPopulation-served field detected:', POPULATION_COLUMN)
    print('Positive population-served coverage:', round(100 * df['population_served'].notna().mean(), 1), '%')
else:
    df['population_served'] = np.nan
    print('\nNo population-served field detected. County median will be the primary observed summary.')


Raw observations: 7,554
Raw columns: ['pws_id', 'pws_name', 'Longitude', 'Latitude', 'date_taken', 'nitrate']
Coordinates outside expected Iowa bounding box: 0

Raw nitrate distribution:
count    7554.000000
mean        5.603663
std         8.469847
min         0.000000
1%          0.000000
5%          0.000000
25%         0.200000
50%         3.000000
75%         8.000000
95%        17.365000
99%        44.000000
max       116.000000
Name: nitrate, dtype: float64
Raw nitrate >=10 mg/L: 1336

No population-served field detected. County median will be the primary observed summary.


In [ ]:
# ============================================================
# 2. LOAD IOWA COUNTY BOUNDARIES
# ============================================================
us_counties = gpd.read_file(COUNTY_SHP_PATH)
if us_counties.crs is None:
    us_counties = us_counties.set_crs('EPSG:4326', allow_override=True)
else:
    us_counties = us_counties.to_crs('EPSG:4326')

iowa_counties = us_counties.loc[us_counties['STATEFP'].astype(str).str.zfill(2) == '19'].copy()
iowa_counties['county_fips'] = (
    iowa_counties['STATEFP'].astype(str).str.zfill(2)
    + iowa_counties['COUNTYFP'].astype(str).str.zfill(3)
).astype(int)
iowa_counties['countyname'] = iowa_counties['NAME'].astype(str)
iowa_counties = iowa_counties[['county_fips', 'countyname', 'geometry']].copy()

assert iowa_counties['county_fips'].nunique() == 99
print('Iowa counties:', iowa_counties['county_fips'].nunique())


Iowa counties: 99


In [ ]:
# ============================================================
# 3. ASSIGN WATER-SAMPLE COORDINATES TO COUNTIES + DISTANCE QA
# ============================================================
coords = df[['Latitude', 'Longitude']].drop_duplicates().reset_index(drop=True)
coords['coord_id'] = np.arange(len(coords), dtype=np.int64)

points = gpd.GeoDataFrame(
    coords,
    geometry=gpd.points_from_xy(coords['Longitude'], coords['Latitude']),
    crs='EPSG:4326'
)

within = gpd.sjoin(points, iowa_counties, how='left', predicate='within')
within['county_assignment'] = np.where(within['countyname'].notna(), 'within', 'unassigned')
within['county_distance_m'] = 0.0
missing_ids = within.loc[within['countyname'].isna(), 'coord_id'].unique()
matched = within.loc[within['countyname'].notna()].copy()

if len(missing_ids) > 0:
    unmatched = points.loc[points['coord_id'].isin(missing_ids)].to_crs('EPSG:26915')
    counties_proj = iowa_counties.to_crs('EPSG:26915')
    nearest = gpd.sjoin_nearest(
        unmatched, counties_proj, how='left', distance_col='county_distance_m'
    )
    nearest = (
        nearest.sort_values(['coord_id', 'county_distance_m', 'county_fips'])
        .drop_duplicates('coord_id', keep='first')
    )
    nearest['county_assignment'] = 'nearest'
    keep = ['coord_id', 'Latitude', 'Longitude', 'county_fips', 'countyname',
            'county_assignment', 'county_distance_m']
    coord_county = pd.concat([matched[keep], nearest[keep]], ignore_index=True)
else:
    keep = ['coord_id', 'Latitude', 'Longitude', 'county_fips', 'countyname',
            'county_assignment', 'county_distance_m']
    coord_county = matched[keep].copy()

assert coord_county['county_fips'].notna().all()
assert coord_county['county_fips'].isin(iowa_counties['county_fips']).all()

nearest_only = coord_county.loc[coord_county['county_assignment'].eq('nearest')].copy()
coord_qa = pd.DataFrame({
    'metric': ['unique_coordinates', 'assigned_within', 'assigned_nearest',
               'nearest_gt_1km', 'nearest_gt_5km', 'nearest_max_km'],
    'value': [
        len(coord_county),
        int(coord_county['county_assignment'].eq('within').sum()),
        int(coord_county['county_assignment'].eq('nearest').sum()),
        int((nearest_only['county_distance_m'] > 1000).sum()),
        int((nearest_only['county_distance_m'] > 5000).sum()),
        float(nearest_only['county_distance_m'].max() / 1000) if len(nearest_only) else 0.0,
    ]
})
display(coord_qa)
if len(nearest_only):
    display(nearest_only.sort_values('county_distance_m', ascending=False).head(25))

df = df.merge(
    coord_county[['Latitude', 'Longitude', 'county_fips', 'countyname',
                  'county_assignment', 'county_distance_m']],
    on=['Latitude', 'Longitude'], how='left', validate='many_to_one'
)
assert df['county_fips'].notna().all()


,metric,value
0,unique_coordinates,1130.0
1,assigned_within,1130.0
2,assigned_nearest,0.0
3,nearest_gt_1km,0.0
4,nearest_gt_5km,0.0
5,nearest_max_km,0.0


In [ ]:
# ============================================================
# 4. COLLAPSE RAW SAMPLES -> PWS-DAY -> PWS-MONTH -> PWS-QUARTER
# ============================================================
pws_day = (
    df.groupby(['county_fips', 'countyname', 'pws_id', 'date_taken'], as_index=False)
    .agg(
        pws_name=('pws_name', 'first') if 'pws_name' in df.columns else ('pws_id', 'first'),
        nitrate_mean=('nitrate', 'mean'),
        nitrate_median=('nitrate', 'median'),
        nitrate_max=('nitrate', 'max'),
        n_raw_samples=('nitrate', 'size'),
        Latitude=('Latitude', 'median'),
        Longitude=('Longitude', 'median'),
        population_served=('population_served', 'median'),
    )
)
pws_day['year'] = pws_day['date_taken'].dt.year
pws_day['month'] = pws_day['date_taken'].dt.month
pws_day['pws_day_ge10'] = (pws_day['nitrate_max'] >= 10).astype(np.int8)

pws_month = (
    pws_day.groupby(['county_fips', 'countyname', 'pws_id', 'year', 'month'], as_index=False)
    .agg(
        pws_month_mean=('nitrate_mean', 'mean'),
        pws_month_median=('nitrate_median', 'median'),
        pws_month_max=('nitrate_max', 'max'),
        pws_month_any_ge10=('pws_day_ge10', 'max'),
        n_test_days=('date_taken', 'nunique'),
        population_served=('population_served', 'median'),
    )
)

pws_month_panel = pws_month.loc[
    pws_month['year'].between(PANEL_START_YEAR, PANEL_END_YEAR)
].copy()
pws_month_panel['quarter'] = ((pws_month_panel['month'] - 1) // 3 + 1).astype(np.int8)

pws_quarter = (
    pws_month_panel.groupby(
        ['county_fips', 'countyname', 'pws_id', 'year', 'quarter'], as_index=False
    )
    .agg(
        pws_q_mean=('pws_month_mean', 'mean'),
        pws_q_median=('pws_month_median', 'median'),
        pws_q_max=('pws_month_max', 'max'),
        pws_q_any_ge10=('pws_month_any_ge10', 'max'),
        months_observed=('month', 'nunique'),
        n_test_days=('n_test_days', 'sum'),
        population_served=('population_served', 'median'),
    )
)

pop_coverage = pws_quarter['population_served'].notna().mean()
USE_POPULATION_WEIGHTS = bool(
    POPULATION_COLUMN is not None and pop_coverage >= POPULATION_WEIGHT_COVERAGE_MIN
)
print(f'PWS-quarter population-served coverage: {100*pop_coverage:.1f}%')
print('Use population-served weighting for PRIMARY observed county exposure:', USE_POPULATION_WEIGHTS)


PWS-quarter population-served coverage: 0.0%
Use population-served weighting for PRIMARY observed county exposure: False


In [ ]:
# ============================================================
# 5. COLLAPSE PWS-QUARTER -> COUNTY-QUARTER
# ============================================================
base_agg = (
    pws_quarter.groupby(['county_fips', 'countyname', 'year', 'quarter'], as_index=False)
    .agg(
        nitrate_median=('pws_q_median', 'median'),
        nitrate_mean=('pws_q_mean', 'mean'),
        nitrate_max=('pws_q_max', 'max'),
        any_pws_ge10=('pws_q_any_ge10', 'max'),
        months_observed=('months_observed', 'max'),
        n_pws=('pws_id', 'nunique'),
        n_test_days=('n_test_days', 'sum'),
    )
)

def weighted_pws_mean(group):
    valid = group['population_served'].notna() & (group['population_served'] > 0) & group['pws_q_mean'].notna()
    if valid.sum() == 0:
        return np.nan
    return np.average(
        group.loc[valid, 'pws_q_mean'].to_numpy(dtype=float),
        weights=group.loc[valid, 'population_served'].to_numpy(dtype=float)
    )

pop_weighted = (
    pws_quarter.groupby(['county_fips', 'countyname', 'year', 'quarter'])
    .apply(weighted_pws_mean, include_groups=False)
    .rename('nitrate_pop_weighted_mean')
    .reset_index()
)
county_quarter_water = base_agg.merge(
    pop_weighted, on=['county_fips', 'countyname', 'year', 'quarter'], how='left', validate='one_to_one'
)

if USE_POPULATION_WEIGHTS:
    county_quarter_water['nitrate_observed_primary'] = county_quarter_water['nitrate_pop_weighted_mean']
    PRIMARY_OBSERVED_METHOD = 'population-served weighted mean across PWS-quarter means'
else:
    county_quarter_water['nitrate_observed_primary'] = county_quarter_water['nitrate_median']
    PRIMARY_OBSERVED_METHOD = 'unweighted median across PWS-quarter medians'

county_quarter_water['quarter_start'] = pd.PeriodIndex(
    year=county_quarter_water['year'], quarter=county_quarter_water['quarter'], freq='Q'
).start_time
county_quarter_water['quarter_end'] = pd.PeriodIndex(
    year=county_quarter_water['year'], quarter=county_quarter_water['quarter'], freq='Q'
).end_time.normalize()
county_quarter_water['county_primary_ge10'] = (
    county_quarter_water['nitrate_observed_primary'] >= 10
).astype(np.int8)
county_quarter_water['county_primary_ge5'] = (
    county_quarter_water['nitrate_observed_primary'] >= 5
).astype(np.int8)

print('Primary observed county-quarter method:', PRIMARY_OBSERVED_METHOD)
print('Observed county-quarters:', len(county_quarter_water))
print(county_quarter_water['nitrate_observed_primary'].describe())


Primary observed county-quarter method: unweighted median across PWS-quarter medians
Observed county-quarters: 1707
count    1707.000000
mean        4.001313
std         4.765856
min         0.000000
25%         0.220000
50%         3.000000
75%         5.890000
max        49.000000
Name: nitrate_observed_primary, dtype: float64


/tmp/ipykernel_512/3014907233.py:43: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  county_quarter_water['quarter_start'] = pd.PeriodIndex(
/tmp/ipykernel_512/3014907233.py:46: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  county_quarter_water['quarter_end'] = pd.PeriodIndex(


In [ ]:
# ============================================================
# 6. COMPLETE COUNTY-QUARTER GRID
# ============================================================
county_list = iowa_counties[['county_fips', 'countyname']].drop_duplicates().reset_index(drop=True)
quarters = pd.DataFrame({
    'quarter_start': pd.date_range(
        start=f'{PANEL_START_YEAR}-01-01', end=f'{PANEL_END_YEAR}-10-01', freq='QS'
    )
})
quarters['year'] = quarters['quarter_start'].dt.year
quarters['quarter'] = quarters['quarter_start'].dt.quarter
quarters['quarter_end'] = quarters['quarter_start'] + pd.offsets.QuarterEnd(startingMonth=3)
county_list['_key'] = 1
quarters['_key'] = 1
county_quarter_grid = county_list.merge(quarters, on='_key').drop(columns='_key')
assert len(county_quarter_grid) == 99 * N_QUARTERS_PER_COUNTY

county_quarter_full = county_quarter_grid.merge(
    county_quarter_water.drop(columns=['quarter_end']),
    on=['county_fips', 'countyname', 'year', 'quarter', 'quarter_start'],
    how='left', validate='one_to_one'
)
county_quarter_full['observed_quarter'] = county_quarter_full['nitrate_observed_primary'].notna().astype(np.int8)

coverage_by_year = (
    county_quarter_full.groupby('year', as_index=False)
    .agg(
        county_quarters=('observed_quarter', 'size'),
        observed_county_quarters=('observed_quarter', 'sum')
    )
)
coverage_by_year['pct_observed'] = 100 * coverage_by_year['observed_county_quarters'] / coverage_by_year['county_quarters']
display(coverage_by_year)


,year,county_quarters,observed_county_quarters,pct_observed
0,1982,396,60,15.151515
1,1983,396,265,66.919192
2,1984,396,237,59.848485
3,1985,396,272,68.686869
4,1986,396,286,72.222222
5,1987,396,248,62.626263
6,1988,396,339,85.606061


In [ ]:
# ============================================================
# 7. COUNTY CENTROID DISTANCE MATRIX
# ============================================================
iowa_proj = iowa_counties.to_crs('EPSG:26915').copy()
iowa_proj['centroid'] = iowa_proj.geometry.centroid
county_centroids = pd.DataFrame({
    'county_fips': iowa_proj['county_fips'].to_numpy(),
    'x': iowa_proj['centroid'].x.to_numpy(),
    'y': iowa_proj['centroid'].y.to_numpy(),
})
coords_xy = county_centroids[['x', 'y']].to_numpy()
distance_matrix = cdist(coords_xy, coords_xy) / 1000.0
county_ids = county_centroids['county_fips'].astype(int).tolist()
distance_df = pd.DataFrame(distance_matrix, index=county_ids, columns=county_ids)


In [ ]:
# ============================================================
# 8. LEAKAGE-SAFE SPATIAL PREDICTOR
# ============================================================
def spatial_predictor_from_training(
    training_data,
    target_data,
    distance_df,
    value_col='nitrate_observed_primary',
    max_distance_km=100,
    power=1,
    fallback_nearest_n=5,
):
    """Same-quarter inverse-distance exposure using TRAINING observations only.

    For a training target, its own county-quarter is excluded. For a held-out
    target, the held-out nitrate never enters the predictor. If no observed
    county lies within max_distance_km, use the nearest observed counties in
    that quarter as a transparent fallback.
    """
    train = training_data.loc[training_data[value_col].notna(),
                              ['county_fips', 'year', 'quarter', value_col]].copy()
    grouped = {}
    for key, g in train.groupby(['year', 'quarter']):
        grouped[key] = (
            g['county_fips'].astype(int).to_numpy(),
            g[value_col].astype(float).to_numpy(),
        )

    out = np.full(len(target_data), np.nan, dtype=float)
    for j, row in enumerate(target_data[['county_fips', 'year', 'quarter']].itertuples(index=False)):
        key = (int(row.year), int(row.quarter))
        if key not in grouped:
            continue
        counties, values = grouped[key]
        target_county = int(row.county_fips)
        keep_other = counties != target_county
        counties2 = counties[keep_other]
        values2 = values[keep_other]
        if len(counties2) == 0:
            continue
        d = distance_df.loc[target_county, counties2].to_numpy(dtype=float)
        near = (d > 0) & (d <= max_distance_km)
        if near.sum() == 0:
            order = np.argsort(d)
            take = order[:min(fallback_nearest_n, len(order))]
            d_use = d[take]
            v_use = values2[take]
        else:
            d_use = d[near]
            v_use = values2[near]
        weights = 1 / np.power(d_use, power)
        out[j] = np.average(v_use, weights=weights)
    return out

observed_data = county_quarter_full.loc[
    county_quarter_full['nitrate_observed_primary'].notna()
].copy()
county_quarter_full['spatial_nitrate'] = spatial_predictor_from_training(
    observed_data, county_quarter_full, distance_df
)
print(county_quarter_full['spatial_nitrate'].describe())
if county_quarter_full['spatial_nitrate'].isna().any():
    raise ValueError('Spatial predictor missing for at least one county-quarter.')


count    2772.000000
mean        3.749332
std         2.386010
min         0.000000
25%         2.121517
50%         3.446824
75%         5.048957
max        16.197369
Name: spatial_nitrate, dtype: float64


In [ ]:
# ============================================================
# 9. FIT FINAL SPATIOTEMPORAL IMPUTATION MODEL
# ============================================================
county_quarter_full['time_id'] = (
    county_quarter_full['year'].astype(str) + '_Q' + county_quarter_full['quarter'].astype(str)
)
county_quarter_full['ln_nitrate'] = np.log1p(county_quarter_full['nitrate_observed_primary'])
county_quarter_full['ln_spatial_nitrate'] = np.log1p(county_quarter_full['spatial_nitrate'])

model_data = county_quarter_full.loc[
    county_quarter_full['nitrate_observed_primary'].notna()
].copy()
final_model = smf.ols(
    'ln_nitrate ~ C(county_fips) + C(time_id) + ln_spatial_nitrate',
    data=model_data
).fit()
print(final_model.summary())


                            OLS Regression Results                            
Dep. Variable:             ln_nitrate   R-squared:                       0.536
Model:                            OLS   Adj. R-squared:                  0.500
Method:                 Least Squares   F-statistic:                     14.51
Date:                Thu, 03 Sep 2026   Prob (F-statistic):          4.62e-186
Time:                        12:53:48   Log-Likelihood:                -1595.4
No. Observations:                1707   AIC:                             3445.
Df Residuals:                    1580   BIC:                             4136.
Df Model:                         126                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

In [ ]:
# ============================================================
# 10. LEAKAGE-SAFE CROSS-VALIDATION
# ============================================================
def categories_supported(train, test):
    return (
        set(test['county_fips']).issubset(set(train['county_fips']))
        and set(test['time_id']).issubset(set(train['time_id']))
    )

def evaluate_split(train, test, model_name):
    if len(test) == 0 or not categories_supported(train, test):
        return None
    train = train.copy()
    test = test.copy()
    if model_name == 'County + Time FE':
        formula = 'ln_nitrate ~ C(county_fips) + C(time_id)'
    elif model_name == 'County + Time FE + Log Spatial':
        train['spatial_cv'] = spatial_predictor_from_training(train, train, distance_df)
        test['spatial_cv'] = spatial_predictor_from_training(train, test, distance_df)
        if train['spatial_cv'].isna().any() or test['spatial_cv'].isna().any():
            return None
        train['ln_spatial_cv'] = np.log1p(train['spatial_cv'])
        test['ln_spatial_cv'] = np.log1p(test['spatial_cv'])
        formula = 'ln_nitrate ~ C(county_fips) + C(time_id) + ln_spatial_cv'
    else:
        raise ValueError(model_name)

    fit = smf.ols(formula, data=train).fit()
    pred_log = fit.predict(test)
    train_pred = fit.predict(train)
    smear = np.mean(np.exp(train['ln_nitrate'].to_numpy() - train_pred.to_numpy()))
    pred = np.exp(pred_log.to_numpy()) * smear - 1
    obs = test['nitrate_observed_primary'].to_numpy(dtype=float)
    return {
        'MAE': mean_absolute_error(obs, pred),
        'RMSE': np.sqrt(mean_squared_error(obs, pred)),
        'Median_AE': np.median(np.abs(obs - pred)),
        'Correlation': np.corrcoef(obs, pred)[0, 1] if len(obs) > 1 else np.nan,
        'Mean_observed': np.mean(obs),
        'Mean_predicted': np.mean(pred),
        'Bias': np.mean(pred) - np.mean(obs),
        'Pct_negative': 100 * np.mean(pred < 0),
        'Smearing_factor': smear,
        'n_test': len(test),
    }

cv_data = model_data.copy()
cv_data['time_id'] = cv_data['year'].astype(str) + '_Q' + cv_data['quarter'].astype(str)
cv_data['ln_nitrate'] = np.log1p(cv_data['nitrate_observed_primary'])
model_names = ['County + Time FE', 'County + Time FE + Log Spatial']
rng = np.random.default_rng(RANDOM_SEED)

# ---- Random 5-fold ----
random_rows = []
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
for model_name in model_names:
    for fold, (train_idx, test_idx) in enumerate(kf.split(cv_data), start=1):
        res = evaluate_split(cv_data.iloc[train_idx], cv_data.iloc[test_idx], model_name)
        if res is not None:
            random_rows.append({'validation': 'random_5fold', 'model': model_name, 'rep': fold, **res})
random_cv = pd.DataFrame(random_rows)
random_summary = random_cv.groupby('model', as_index=False).mean(numeric_only=True)
print('\nRANDOM 5-FOLD CV')
display(random_summary)

# ---- Blocked contiguous two-quarter holdout within county ----
blocked_rows = []
for rep in range(BLOCKED_CV_REPS):
    holdout = []
    for county, g in cv_data.groupby('county_fips'):
        g = g.sort_values(['year', 'quarter'])
        if len(g) < 4:
            continue
        starts = np.arange(0, len(g) - 1)
        start = int(rng.choice(starts))
        holdout.extend(g.iloc[start:start+2].index.tolist())
    test = cv_data.loc[holdout]
    train = cv_data.drop(index=holdout)
    for model_name in model_names:
        res = evaluate_split(train, test, model_name)
        if res is not None:
            blocked_rows.append({'validation': 'blocked_2quarter', 'model': model_name, 'rep': rep+1, **res})
blocked_cv = pd.DataFrame(blocked_rows)
blocked_summary = blocked_cv.groupby('model', as_index=False).mean(numeric_only=True)
print('\nBLOCKED CV')
display(blocked_summary)

# ---- Artificial missingness pattern validation ----
primary_period_obs = cv_data.loc[cv_data['year'].between(1983, 1988)].copy()
county_counts = primary_period_obs.groupby('county_fips').size()
well_counties = county_counts.loc[county_counts >= WELL_OBSERVED_MIN_QUARTERS_1983_1988].index.tolist()
print('Well-observed counties for missingness validation:', len(well_counties))
missing_rows = []
for retention in MISSINGNESS_RETENTION_RATES:
    for rep in range(MISSINGNESS_REPS):
        holdout = []
        for county in well_counties:
            idx = primary_period_obs.index[primary_period_obs['county_fips'].eq(county)].to_numpy()
            n_keep = max(4, int(np.ceil(len(idx) * retention)))
            n_keep = min(n_keep, len(idx))
            keep_idx = rng.choice(idx, size=n_keep, replace=False)
            holdout.extend(np.setdiff1d(idx, keep_idx).tolist())
        test = cv_data.loc[holdout]
        train = cv_data.drop(index=holdout)
        for model_name in model_names:
            res = evaluate_split(train, test, model_name)
            if res is not None:
                missing_rows.append({
                    'validation': 'artificial_missingness', 'retention_rate': retention,
                    'model': model_name, 'rep': rep+1, **res
                })
missingness_cv = pd.DataFrame(missing_rows)
missingness_summary = (
    missingness_cv.groupby(['retention_rate', 'model'], as_index=False)
    .mean(numeric_only=True)
)
print('\nARTIFICIAL MISSINGNESS VALIDATION')
display(missingness_summary)



RANDOM 5-FOLD CV


,model,rep,MAE,RMSE,Median_AE,Correlation,Mean_observed,Mean_predicted,Bias,Pct_negative,Smearing_factor,n_test
0,County + Time FE,3.0,2.382780,4.109135,1.565288,0.528472,4.000933,4.069902,0.068969,0.468351,1.212867,341.4
1,County + Time FE + Log Spatial,3.0,2.406054,4.215453,1.563926,0.514785,4.078534,4.110912,0.032378,0.658752,1.208368,341.5



BLOCKED CV


,model,rep,MAE,RMSE,Median_AE,Correlation,Mean_observed,Mean_predicted,Bias,Pct_negative,Smearing_factor,n_test
0,County + Time FE,25.5,2.088041,3.627882,1.326355,0.569124,3.251496,3.277803,0.026307,0.626263,1.214908,198.0
1,County + Time FE + Log Spatial,25.5,2.070740,3.590559,1.286492,0.578925,3.251496,3.270450,0.018954,0.949495,1.212193,198.0


Well-observed counties for missingness validation: 51

ARTIFICIAL MISSINGNESS VALIDATION


,retention_rate,model,rep,MAE,RMSE,Median_AE,Correlation,Mean_observed,Mean_predicted,Bias,Pct_negative,Smearing_factor,n_test
0,0.25,County + Time FE,15.5,2.833888,4.320777,2.032533,0.469895,5.014553,5.327256,0.312703,0.255183,1.227506,836.0
1,0.25,County + Time FE + Log Spatial,15.5,2.828082,4.308070,2.022350,0.472267,5.014553,5.325483,0.310930,0.259171,1.226270,836.0
2,0.50,County + Time FE,15.5,2.705702,4.193109,1.940614,0.489080,4.984698,5.370402,0.385703,0.090253,1.222977,554.0
3,0.50,County + Time FE + Log Spatial,15.5,2.689604,4.159989,1.927338,0.496036,4.984698,5.362521,0.377823,0.108303,1.220935,554.0
4,0.75,County + Time FE,15.5,2.594043,4.017270,1.870785,0.508862,5.003526,5.350360,0.346834,0.075188,1.221855,266.0
5,0.75,County + Time FE + Log Spatial,15.5,2.574556,3.973425,1.856304,0.518754,5.003526,5.340426,0.336900,0.162907,1.218857,266.0


In [ ]:
# ============================================================
# 11. FINAL PREDICTIONS AND COMPLETED EXPOSURE SERIES
# ============================================================
train_pred_log = final_model.predict(model_data)
resid = model_data['ln_nitrate'].to_numpy() - train_pred_log.to_numpy()
smearing_factor = np.mean(np.exp(resid))
print('Final Duan smearing factor:', smearing_factor)

county_quarter_full['ln_nitrate_predicted'] = final_model.predict(county_quarter_full)
county_quarter_full['nitrate_predicted_raw'] = (
    np.exp(county_quarter_full['ln_nitrate_predicted']) * smearing_factor - 1
)
county_quarter_full['nitrate_observed'] = county_quarter_full['nitrate_observed_primary']
county_quarter_full['exposure_source'] = np.where(
    county_quarter_full['nitrate_observed'].notna(), 'observed', 'predicted'
)
county_quarter_full['nitrate_imputed'] = county_quarter_full['exposure_source'].eq('predicted').astype(np.int8)
county_quarter_full['nitrate_complete'] = county_quarter_full['nitrate_observed']
missing = county_quarter_full['nitrate_complete'].isna()
county_quarter_full.loc[missing, 'nitrate_complete'] = county_quarter_full.loc[missing, 'nitrate_predicted_raw']

negative_imputed = county_quarter_full['nitrate_imputed'].eq(1) & county_quarter_full['nitrate_complete'].lt(0)
print('Negative imputed predictions before flooring:', int(negative_imputed.sum()))
county_quarter_full.loc[negative_imputed, 'nitrate_complete'] = 0.0

print('Percent county-quarters imputed:', round(100 * county_quarter_full['nitrate_imputed'].mean(), 1))
print('\nObserved primary nitrate:')
print(county_quarter_full.loc[county_quarter_full['exposure_source'].eq('observed'), 'nitrate_complete'].describe())
print('\nPredicted nitrate:')
print(county_quarter_full.loc[county_quarter_full['exposure_source'].eq('predicted'), 'nitrate_complete'].describe())


Final Duan smearing factor: 1.2141269686784701
Negative imputed predictions before flooring: 109
Percent county-quarters imputed: 38.4

Observed primary nitrate:
count    1707.000000
mean        4.001313
std         4.765856
min         0.000000
25%         0.220000
50%         3.000000
75%         5.890000
max        49.000000
Name: nitrate_complete, dtype: float64

Predicted nitrate:
count    1065.000000
mean        1.672093
std         1.727713
min         0.000000
25%         0.515089
50%         1.152594
75%         2.353322
max        14.379855
Name: nitrate_complete, dtype: float64


In [ ]:
# ============================================================
# 12. FINAL PANEL + QA OUTPUTS
# ============================================================
final_cols = [
    'county_fips', 'countyname', 'year', 'quarter', 'quarter_start', 'quarter_end',
    'nitrate_complete', 'nitrate_observed', 'nitrate_predicted_raw',
    'nitrate_imputed', 'exposure_source', 'spatial_nitrate',
    'nitrate_median', 'nitrate_mean', 'nitrate_max', 'nitrate_pop_weighted_mean',
    'months_observed', 'n_pws', 'n_test_days', 'any_pws_ge10',
    'county_primary_ge10', 'county_primary_ge5',
]
for c in final_cols:
    if c not in county_quarter_full.columns:
        county_quarter_full[c] = np.nan

final_county_quarter = (
    county_quarter_full[final_cols]
    .sort_values(['county_fips', 'year', 'quarter'])
    .reset_index(drop=True)
)
assert len(final_county_quarter) == 99 * N_QUARTERS_PER_COUNTY
assert final_county_quarter['county_fips'].nunique() == 99
assert final_county_quarter['nitrate_complete'].notna().all()
assert (final_county_quarter['nitrate_complete'] >= 0).all()

# QA comparing observed vs imputed distributions.
source_qa = (
    final_county_quarter.groupby('exposure_source')['nitrate_complete']
    .describe(percentiles=[.1, .25, .5, .75, .9])
    .reset_index()
)
display(source_qa)


,exposure_source,count,mean,std,min,10%,25%,50%,75%,90%,max
0,observed,1707.0,4.001313,4.765856,0.0,0.0,0.220000,3.000000,5.890000,8.890000,49.000000
1,predicted,1065.0,1.672093,1.727713,0.0,0.0,0.515089,1.152594,2.353322,3.773267,14.379855


In [ ]:
# ============================================================
# 13. SAVE
# ============================================================
PANEL_CSV = os.path.join(OUTPUT_DIR, 'final_county_quarter_complete_1982_1988_v3.csv')
PANEL_PARQUET = os.path.join(OUTPUT_DIR, 'final_county_quarter_complete_1982_1988_v3.parquet')
final_county_quarter.to_csv(PANEL_CSV, index=False)
try:
    final_county_quarter.to_parquet(PANEL_PARQUET, index=False)
except Exception as exc:
    print('Parquet save skipped:', exc)

pws_quarter.to_csv(os.path.join(OUTPUT_DIR, 'pws_quarter_1982_1988_v3.csv'), index=False)
coverage_by_year.to_csv(os.path.join(OUTPUT_DIR, 'water_coverage_by_year_v3.csv'), index=False)
coord_qa.to_csv(os.path.join(OUTPUT_DIR, 'coordinate_assignment_qa_v3.csv'), index=False)
nearest_only.sort_values('county_distance_m', ascending=False).to_csv(
    os.path.join(OUTPUT_DIR, 'nearest_county_assignments_v3.csv'), index=False
)
random_cv.to_csv(os.path.join(OUTPUT_DIR, 'validation_random_cv_v3.csv'), index=False)
blocked_cv.to_csv(os.path.join(OUTPUT_DIR, 'validation_blocked_cv_v3.csv'), index=False)
missingness_cv.to_csv(os.path.join(OUTPUT_DIR, 'validation_missingness_pattern_v3.csv'), index=False)
source_qa.to_csv(os.path.join(OUTPUT_DIR, 'observed_vs_predicted_distribution_v3.csv'), index=False)

metadata = pd.DataFrame({
    'item': [
        'panel_start_year', 'primary_cohort_start_year', 'panel_end_year',
        'primary_observed_method', 'population_field_detected',
        'population_weight_coverage_threshold', 'imputation_model',
        'note_1982'
    ],
    'value': [
        PANEL_START_YEAR, PRIMARY_COHORT_START_YEAR, PANEL_END_YEAR,
        PRIMARY_OBSERVED_METHOD, str(POPULATION_COLUMN),
        POPULATION_WEIGHT_COVERAGE_MIN,
        'log1p nitrate ~ county FE + year-quarter FE + log same-quarter spatial nitrate; Duan smearing',
        '1982 retained to preserve the validated V3 water reconstruction; analytic T1 cohort begins at gestational start 1983-01-01'
    ]
})
metadata.to_csv(os.path.join(OUTPUT_DIR, 'water_panel_metadata_v3.csv'), index=False)

print('File 1 V3 complete.')
print('Primary water panel:', PANEL_CSV)


File 1 V3 complete.
Primary water panel: /content/drive/MyDrive/plos-update-v3/1-water/final_county_quarter_complete_1982_1988_v3.csv
